# 03e · 익명 test 추론과 예측 봉인

이 파일은 DCC 2026 작업 노트북의 **공개용 보존본**입니다. 아래 원래 셀과 텍스트 출력은 저장된 실험 기록이며, 이번 공개 작업에서 모델 학습·추론·평가를 다시 실행하지 않았습니다.

- **목적:** 선택 seal과 익명 test 이미지만 읽어 예측을 만들고 정답 접근 전에 예측 파일을 봉인합니다.
- **입력:** 03D의 full selection, 익명 test image seal과 선택된 모델 가중치
- **출력:** 익명 test 예측과 prediction seal
- **실제 저장된 실행 상태:** 저장 출력은 anonymous test contract PASS 30,809, TEST PREDICTIONS FROZEN, truth accessed=False입니다. 이 노트북은 test RMSE를 계산하지 않습니다. 이어지는 03F는 별도 복구 코드이며 저장 출력이 없습니다.

**실행 전:** 공개 저장소에는 원천 이미지·라벨 ZIP, private test 연결표·정답, 모델 가중치, 원래 GCS 산출물이 없습니다. `YOUR_GCP_PROJECT_ID`, `YOUR_GCS_BUCKET`과 필요한 데이터·권한을 직접 설정해야 하며, 원래 봉인 파일의 generation·SHA-256·config 검증도 충족해야 합니다. placeholder만 바꾼다고 과거 산출물이 재생성되거나 모든 gate가 통과하는 것은 아닙니다.

**공개본 변경:** 개인 프로젝트·버킷·계정·로컬 데이터 경로를 placeholder로 바꾸고 Colab/Drive/작성자/widget metadata를 제거했습니다. 코드 셀의 순서와 계산·무결성 검증 로직은 보존했습니다. 모든 내장 image/HTML/JavaScript/기타 binary 출력과 markdown 첨부 이미지를 제거하고, 숫자·표·오류를 포함한 텍스트 출력은 식별자를 가린 뒤 보존했습니다. 사진이 없는 것은 해당 셀을 실행하지 않았다는 뜻이 아닙니다. 원본 code cell의 `execution_count`는 모두 `null`이므로 이 값만으로 실행 여부를 판단하지 않습니다.

[GitHub 코드](https://github.com/Pirite08/dcc-2026-water-level-estimation/blob/main/notebooks/03e_test_prediction.ipynb) · [Colab에서 열기](https://colab.research.google.com/github/Pirite08/dcc-2026-water-level-estimation/blob/main/notebooks/03e_test_prediction.ipynb) · [실행 관계와 상태](README.md) · [원본 이름·SHA-256 대응표](../docs/notebook-source-map.csv)

---

아래는 비공개 식별자를 치환한 원래 노트북입니다. 원래 안내의 예정 단계·성공 기준은 위의 실제 저장 상태와 구분해 읽으십시오.


# DCC 2026 Mission 3 — 03E. 동결 승자·유형 라우터의 test 추론과 예측 봉인

이 노트북은 다음 정보만 읽습니다.

- 익명 test image shard: `sample_id`, 이미지 byte, 파일명 첫 문자에서 미리 만든 type
- 03D에서 동결한 승자 checkpoint, 유형 라우팅과 전처리 규칙

`test_truth_v3`, 원래 stem, ZIP 경로, annotation, obstruction은 읽지 않습니다. 예측 CSV와
diagnostics를 먼저 GCS에 create-only로 게시하고 SHA-256 seal을 만든 뒤 종료합니다.


## clean GPU runtime

선택된 후보의 full-frame·YOLO+ROI component를 실행할 inference package를 설치합니다.
새 GPU runtime에서 첫 셀부터 실행하십시오. 이 노트북은 `m3-selection-full-router-v2`의
`MODEL_DEPLOYMENT_FROZEN` seal만 허용하며 pilot checkpoint는 즉시 차단합니다.


In [ ]:
%pip -q install "ultralytics==8.4.121" "google-cloud-storage>=2.19,<4" "pandas>=2.2,<3" "pyarrow>=18,<24" "webdataset>=1.0.2,<2"


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.6/45.6 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 53.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.0/75.0 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.0/66.0 kB 6.4 MB/s eta 0:00:00


In [ ]:
import gc
import hashlib
import json
import math
import os
import platform
import shutil
import tarfile
import time
from collections import defaultdict
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import ultralytics
import webdataset as wds
from google.colab import auth
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms
from ultralytics import YOLO

PROJECT_ID = "YOUR_GCP_PROJECT_ID"
BUCKET_NAME = "YOUR_GCS_BUCKET"
SELECTION_ID = "m3-selection-full-router-v2"
PREDICTION_RUN_ID = "m3-test-prediction-router-full-v2"
EXPECTED_TEST_N = 30_809
EXPECTED_TEST_TYPES = {
    "A": 10_522, "B": 3_727, "C": 9_194, "E": 2_351,
    "F": 878, "G": 1_098, "H": 1_031, "S": 2_008,
}
TYPE_ORDER = list("ABCEFGHS")
BATCH_SIZE = 64
NUM_WORKERS = 2

LOCAL_ROOT = Path("/content/dcc_mission3_03e")
LOCAL_INPUT = LOCAL_ROOT / "input"
LOCAL_SHARDS = LOCAL_ROOT / "test_shards"
LOCAL_IMAGES = LOCAL_ROOT / "test_images"
LOCAL_ROUTED_DETECTOR_IMAGES = LOCAL_ROOT / "detector_images"
LOCAL_OUTPUT = LOCAL_ROOT / "output"
LOCAL_META = LOCAL_ROOT / "meta"
for path in [
    LOCAL_INPUT, LOCAL_SHARDS, LOCAL_IMAGES, LOCAL_ROUTED_DETECTOR_IMAGES,
    LOCAL_OUTPUT, LOCAL_META,
]:
    path.mkdir(parents=True, exist_ok=True)

SELECTION_OBJECT = f"dcc/mission3/selection/{SELECTION_ID}/model_selection_seal.json"
OUTPUT_PREFIX = f"dcc/inference/mission3/predictions/{PREDICTION_RUN_ID}"
# 예측/진단 업로드 뒤 final seal 전에 runtime이 끊겨도 새 attempt로 재실행할 수 있습니다.
# final prediction seal만 고정 경로에 create-only로 게시합니다.
ATTEMPT_ID = (
    datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    + "-" + os.urandom(4).hex()
)
ATTEMPT_PREFIX = f"{OUTPUT_PREFIX}/attempts/{ATTEMPT_ID}"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if DEVICE.type != "cuda":
    raise RuntimeError("test 추론에는 GPU runtime을 사용하십시오")
print("GPU:", torch.cuda.get_device_name(0))
print("torch:", torch.__version__, "torchvision:", torchvision.__version__)
print("ultralytics:", ultralytics.__version__)


Creating new Ultralytics Settings v0.0.7 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
GPU: Tesla T4
torch: 2.11.0+cu128 torchvision: 0.26.0+cu128
ultralytics: 8.4.121


In [ ]:
from google.api_core.exceptions import NotFound, PreconditionFailed
from google.cloud import storage

def sha256_file(path, chunk_size=8 * 1024 * 1024):
    digest = hashlib.sha256()
    with open(path, "rb") as stream:
        while True:
            chunk = stream.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()

def canonical_sha256(value):
    raw = json.dumps(
        value, ensure_ascii=False, sort_keys=True, separators=(",", ":"),
        allow_nan=False,
    ).encode("utf-8")
    return hashlib.sha256(raw).hexdigest()

def canonical_target_sha256(frame, target_column):
    """원본 float64와 DataLoader float32가 같은 수위를 같은 byte 계약으로 해시합니다."""
    required = {"stem", "type", target_column}
    missing = required - set(frame.columns)
    if missing:
        raise AssertionError(f"target hash 필수 열 누락: {sorted(missing)}")
    ordered = frame.sort_values("stem", kind="mergesort")
    records = [
        {
            "stem": str(getattr(row, "stem")),
            "type": str(getattr(row, "type")),
            "target_m_float32": float(np.float32(float(getattr(row, target_column)))),
        }
        for row in ordered.itertuples(index=False)
    ]
    return canonical_sha256(records)

def parse_gs_uri(uri):
    if not str(uri).startswith("gs://"):
        raise ValueError(f"GCS URI가 아닙니다: {uri}")
    bucket_name, object_name = str(uri)[5:].split("/", 1)
    return bucket_name, object_name

def download_object(bucket, object_name, local_path, generation=None):
    local_path = Path(local_path)
    local_path.parent.mkdir(parents=True, exist_ok=True)
    blob = bucket.blob(object_name, generation=generation)
    kwargs = {"checksum": "auto"}
    if generation is not None:
        kwargs["if_generation_match"] = int(generation)
    blob.download_to_filename(str(local_path), **kwargs)
    return local_path

def upload_create_only(bucket, local_path, object_name, artifact_type):
    local_path = Path(local_path)
    file_sha = sha256_file(local_path)
    blob = bucket.blob(object_name)
    blob.metadata = {
        "sha256": file_sha,
        "artifact_type": artifact_type,
    }
    try:
        blob.upload_from_filename(
            str(local_path), if_generation_match=0, checksum="auto"
        )
    except PreconditionFailed:
        blob.reload()
        remote_sha = (blob.metadata or {}).get("sha256")
        remote_artifact_type = (blob.metadata or {}).get("artifact_type")
        if (
            remote_sha != file_sha
            or remote_artifact_type != artifact_type
            or int(blob.size or -1) != local_path.stat().st_size
        ):
            raise RuntimeError(
                f"create-only 충돌: gs://{bucket.name}/{object_name}"
            )
    blob.reload()
    if (
        (blob.metadata or {}).get("sha256") != file_sha
        or (blob.metadata or {}).get("artifact_type") != artifact_type
    ):
        raise RuntimeError(f"업로드 SHA metadata 검증 실패: {object_name}")
    return {
        "uri": f"gs://{bucket.name}/{object_name}",
        "generation": int(blob.generation),
        "bytes": int(blob.size),
        "sha256": file_sha,
        "artifact_type": artifact_type,
    }

def upload_json_create_only(bucket, payload, object_name, artifact_type):
    local_path = LOCAL_META / Path(object_name).name
    local_path.parent.mkdir(parents=True, exist_ok=True)
    local_path.write_text(
        json.dumps(
            payload, ensure_ascii=False, sort_keys=True, indent=2, allow_nan=False
        ) + "\n",
        encoding="utf-8",
    )
    return upload_create_only(bucket, local_path, object_name, artifact_type)

def download_json(bucket, object_name, generation=None):
    blob = bucket.blob(object_name, generation=generation)
    kwargs = {}
    if generation is not None:
        kwargs["if_generation_match"] = int(generation)
    return json.loads(blob.download_as_bytes(**kwargs).decode("utf-8"))

def remote_descriptor(bucket, object_name, artifact_type=None):
    blob = bucket.blob(object_name)
    blob.reload()
    metadata = blob.metadata or {}
    file_sha = metadata.get("sha256")
    if not file_sha:
        raise RuntimeError(f"remote object에 sha256 metadata가 없습니다: {object_name}")
    remote_artifact_type = metadata.get("artifact_type")
    if artifact_type is not None and remote_artifact_type != artifact_type:
        raise RuntimeError(
            f"remote artifact_type 불일치: {object_name} "
            f"({remote_artifact_type!r} != {artifact_type!r})"
        )
    return {
        "uri": f"gs://{bucket.name}/{blob.name}",
        "generation": int(blob.generation),
        "bytes": int(blob.size),
        "sha256": file_sha,
        "artifact_type": remote_artifact_type or "unknown",
    }


In [ ]:
TYPE_ORDER = list("ABCEFGHS")
TYPE_TO_INDEX = {name: index for index, name in enumerate(TYPE_ORDER)}
TYPE_RANGES = {
    "A": (0.0, 15.0), "B": (0.0, 9.0),
    "C": (0.0, 15.0), "E": (0.0, 10.0),
    "F": (0.0, 4.0),  "G": (0.0, 4.0),
    "H": (0.0, 20.0), "S": (0.0, 20.0),
}

class Letterbox:
    def __init__(self, size, fill=(114, 114, 114)):
        self.size = int(size)
        self.fill = fill

    def __call__(self, image):
        image = image.convert("RGB")
        width, height = image.size
        scale = min(self.size / width, self.size / height)
        new_size = (
            max(1, int(round(width * scale))),
            max(1, int(round(height * scale))),
        )
        resized = image.resize(new_size, Image.Resampling.BILINEAR)
        canvas = Image.new("RGB", (self.size, self.size), self.fill)
        offset = ((self.size - new_size[0]) // 2, (self.size - new_size[1]) // 2)
        canvas.paste(resized, offset)
        return canvas

def build_transform(size, training=False, augment=False):
    steps = [transforms.Lambda(Letterbox(size))]
    if training and augment:
        steps.extend([
            transforms.ColorJitter(brightness=0.12, contrast=0.12, saturation=0.06),
            transforms.RandomAffine(
                degrees=2.0, translate=(0.02, 0.02), scale=(0.96, 1.04),
                fill=(114, 114, 114),
            ),
        ])
    steps.extend([
        transforms.ToTensor(),
        transforms.Normalize(
            mean=(0.485, 0.456, 0.406),
            std=(0.229, 0.224, 0.225),
        ),
    ])
    return transforms.Compose(steps)

def build_backbone(name, pretrained):
    if name == "resnet18":
        weights = models.ResNet18_Weights.DEFAULT if pretrained else None
        backbone = models.resnet18(weights=weights)
        feature_dim = backbone.fc.in_features
        backbone.fc = nn.Identity()
    elif name == "efficientnet_b0":
        weights = models.EfficientNet_B0_Weights.DEFAULT if pretrained else None
        backbone = models.efficientnet_b0(weights=weights)
        feature_dim = backbone.classifier[1].in_features
        backbone.classifier = nn.Identity()
    elif name == "convnext_tiny":
        weights = models.ConvNeXt_Tiny_Weights.DEFAULT if pretrained else None
        backbone = models.convnext_tiny(weights=weights)
        feature_dim = backbone.classifier[2].in_features
        backbone.classifier[2] = nn.Identity()
    else:
        raise ValueError(f"지원하지 않는 backbone: {name}")
    return backbone, int(feature_dim)

class TypeConditionalRegressor(nn.Module):
    def __init__(self, backbone_name, pretrained=True, dropout=0.15):
        super().__init__()
        self.backbone_name = backbone_name
        self.backbone, feature_dim = build_backbone(backbone_name, pretrained)
        self.type_embedding = nn.Embedding(len(TYPE_ORDER), 32)
        self.head = nn.Sequential(
            nn.Linear(feature_dim + 32, 256),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(256, 1),
        )
        lows = [TYPE_RANGES[name][0] for name in TYPE_ORDER]
        highs = [TYPE_RANGES[name][1] for name in TYPE_ORDER]
        self.register_buffer("type_low", torch.tensor(lows, dtype=torch.float32))
        self.register_buffer("type_high", torch.tensor(highs, dtype=torch.float32))

    def forward(self, images, type_index):
        features = self.backbone(images)
        type_features = self.type_embedding(type_index)
        raw = self.head(torch.cat([features, type_features], dim=1)).squeeze(1)
        normalized = torch.sigmoid(raw)
        low = self.type_low[type_index]
        high = self.type_high[type_index]
        return low + normalized * (high - low)

def regression_loss(pred_m, target_m, type_index):
    lows = pred_m.new_tensor([TYPE_RANGES[name][0] for name in TYPE_ORDER])
    highs = pred_m.new_tensor([TYPE_RANGES[name][1] for name in TYPE_ORDER])
    low = lows[type_index]
    span = highs[type_index] - low
    pred_norm = (pred_m - low) / span
    target_norm = (target_m - low) / span
    robust = F.smooth_l1_loss(pred_norm, target_norm, beta=0.05)
    meter_term = torch.mean(((pred_m - target_m) / 20.0) ** 2)
    return robust + 0.20 * meter_term

def metric_payload(
    frame, truth_column="target_m", prediction_column="pred_m",
    require_all_types=True,
):
    required = {"type", truth_column, prediction_column}
    if required - set(frame.columns):
        raise AssertionError(f"metric 열 누락: {sorted(required - set(frame.columns))}")
    values = frame[[truth_column, prediction_column]].to_numpy(dtype=float)
    if not np.isfinite(values).all():
        raise AssertionError("metric 입력에 non-finite 값이 있습니다")
    error = frame[prediction_column].to_numpy(float) - frame[truth_column].to_numpy(float)
    overall = float(np.sqrt(np.mean(error ** 2)))
    by_type = {}
    present_types = [
        type_name for type_name in TYPE_ORDER
        if frame["type"].eq(type_name).any()
    ]
    if require_all_types and present_types != TYPE_ORDER:
        missing_types = [name for name in TYPE_ORDER if name not in present_types]
        raise AssertionError(f"metric에 유형이 없습니다: {missing_types}")
    for type_name in present_types:
        part = frame.loc[frame["type"].eq(type_name)]
        part_error = part[prediction_column].to_numpy(float) - part[truth_column].to_numpy(float)
        by_type[type_name] = {
            "n": int(len(part)),
            "rmse_m": float(np.sqrt(np.mean(part_error ** 2))),
            "mae_m": float(np.mean(np.abs(part_error))),
            "bias_m": float(np.mean(part_error)),
        }
    weighted_mse = sum(
        item["n"] * item["rmse_m"] ** 2 for item in by_type.values()
    ) / sum(item["n"] for item in by_type.values())
    if not np.isclose(overall ** 2, weighted_mse, rtol=1e-10, atol=1e-12):
        raise AssertionError("전체 RMSE와 유형별 weighted MSE가 일치하지 않습니다")
    return {
        "overall_rmse_m": overall,
        "worst_type_rmse_m": max(item["rmse_m"] for item in by_type.values()),
        "by_type": by_type,
    }


## 1. selection seal과 익명 image seal만 로드

`eval_private` 경로는 이 runtime에서 사용하지 않습니다.


In [ ]:
auth.authenticate_user()
STORAGE_CLIENT = storage.Client(project=PROJECT_ID)
BUCKET = STORAGE_CLIENT.bucket(BUCKET_NAME)
BUCKET.reload()

selection_path = LOCAL_INPUT / "model_selection_seal.json"
download_object(BUCKET, SELECTION_OBJECT, selection_path)
selection_sha256 = sha256_file(selection_path)
selection = json.loads(selection_path.read_text(encoding="utf-8"))
if selection["status"] != "MODEL_DEPLOYMENT_FROZEN":
    raise AssertionError("full deployment seal이 아닙니다")
if selection["training_scope"] != "full":
    raise RuntimeError("pilot checkpoint는 실제 test 추론에 사용할 수 없습니다")
if selection["selection_id"] != SELECTION_ID:
    raise AssertionError("selection ID 불일치")
if selection.get("schema_version") != "dcc-m3-model-selection-seal-v3":
    raise AssertionError("selection seal schema 오류")
selection_assertions = selection.get("assertions", {})
if not selection_assertions or not all(selection_assertions.values()):
    raise AssertionError("selection seal assertion 실패")
winner = selection["winner"]
if selection.get("winner_model_id") != winner.get("model_id"):
    raise AssertionError("selection winner model ID 불일치")
if (
    winner.get("architecture_payload") is not None
    and canonical_sha256(winner["architecture_payload"])
    != winner.get("architecture_fingerprint")
):
    raise AssertionError("winner architecture fingerprint 불일치")
if winner["kind"] not in {"full_frame", "yolo_roi", "type_router"}:
    raise RuntimeError(f"지원하지 않는 winner kind: {winner['kind']}")
ROUTING_CONTRACT = None
ROUTING_SHA256 = None
ROUTE_COUNTS = None
if winner["kind"] == "type_router":
    route_by_type = dict(winner.get("route_by_type", {}))
    if winner.get("routing_key") != "filename_first_character":
        raise AssertionError("router routing key 오류")
    if set(route_by_type) != set(TYPE_ORDER):
        raise AssertionError("router type 집합 오류")
    if not set(route_by_type.values()).issubset({"full_frame", "yolo_roi"}):
        raise AssertionError("router component 값 오류")
    components = winner.get("components", {})
    if set(components) != {"full_frame", "yolo_roi"}:
        raise AssertionError("router component 계약 오류")
    if {
        key: components[key]["model_id"] for key in ["full_frame", "yolo_roi"]
    } != winner.get("component_model_ids"):
        raise AssertionError("router component model ID 불일치")
    architecture_payload = winner.get("architecture_payload", {})
    expected_component_fingerprints = {
        key: components[key]["architecture_fingerprint"]
        for key in ["full_frame", "yolo_roi"]
    }
    if architecture_payload.get("schema_version") != "dcc-m3-type-router-architecture-v1":
        raise AssertionError("router architecture payload schema 오류")
    if architecture_payload.get("kind") != winner["kind"]:
        raise AssertionError("router architecture kind 불일치")
    if architecture_payload.get("routing_key") != winner["routing_key"]:
        raise AssertionError("router architecture routing key 불일치")
    if architecture_payload.get("route_by_type") != route_by_type:
        raise AssertionError("router architecture route map 불일치")
    if architecture_payload.get("component_model_ids") != winner["component_model_ids"]:
        raise AssertionError("router architecture component model ID 불일치")
    if (
        architecture_payload.get("component_architecture_fingerprints")
        != expected_component_fingerprints
    ):
        raise AssertionError("router architecture component fingerprint 불일치")
    ROUTING_CONTRACT = {
        "routing_key": winner["routing_key"],
        "route_by_type": route_by_type,
        "component_model_ids": winner["component_model_ids"],
    }
    ROUTING_SHA256 = canonical_sha256(ROUTING_CONTRACT)
    ROUTE_COUNTS = {
        component_name: int(sum(
            EXPECTED_TEST_TYPES[type_name]
            for type_name, routed in route_by_type.items()
            if routed == component_name
        ))
        for component_name in ["full_frame", "yolo_roi"]
    }
    if sum(ROUTE_COUNTS.values()) != EXPECTED_TEST_N:
        raise AssertionError("router 예상 test count 오류")

DATASET_VERSION = selection["dataset_version"]
INFERENCE_SEAL_OBJECT = (
    f"dcc/inference/mission3/{DATASET_VERSION}/test_image_seal.json"
)
inference_seal_path = LOCAL_INPUT / "test_image_seal.json"
download_object(BUCKET, INFERENCE_SEAL_OBJECT, inference_seal_path)
inference_seal_sha256 = sha256_file(inference_seal_path)
inference_seal_descriptor = remote_descriptor(
    BUCKET, INFERENCE_SEAL_OBJECT, "m3_inference_image_seal"
)
if inference_seal_descriptor["sha256"] != inference_seal_sha256:
    raise AssertionError("inference image seal descriptor SHA 불일치")
inference_seal = json.loads(inference_seal_path.read_text(encoding="utf-8"))
if inference_seal["status"] != "INFERENCE_IMAGES_COMPLETE":
    raise AssertionError("test image seal 상태 오류")
if inference_seal["dataset_version"] != DATASET_VERSION:
    raise AssertionError("selection과 inference dataset version 불일치")
inference_assertions = inference_seal.get("assertions", {})
if not inference_assertions or not all(inference_assertions.values()):
    raise AssertionError("inference image seal assertion 실패")
split_seal_sha256 = str(inference_seal.get("split_seal_sha256", ""))
if len(split_seal_sha256) != 64 or any(ch not in "0123456789abcdef" for ch in split_seal_sha256):
    raise AssertionError("inference split seal SHA 형식 오류")
dataset_seal_descriptor = inference_seal.get("dataset_seal")
if not dataset_seal_descriptor:
    raise AssertionError("inference seal에 pinned dataset seal descriptor가 없습니다")
if dataset_seal_descriptor["sha256"] != inference_seal["dataset_seal_sha256"]:
    raise AssertionError("inference dataset seal SHA 불일치")

requested_objects = [SELECTION_OBJECT, INFERENCE_SEAL_OBJECT]
forbidden_tokens = ["eval_private", "test_truth", "splits_v3", "raw/source"]
if any(token in name for token in forbidden_tokens for name in requested_objects):
    raise AssertionError("test inference에 금지 object가 포함됐습니다")
if (LOCAL_ROOT / "eval_private").exists():
    raise AssertionError("local eval_private 경로가 존재합니다")

checkpoint_value = winner.get(
    "checkpoint_bundle", winner.get("checkpoint")
)
INFERENCE_CONTRACT = {
    "schema_version": "dcc-m3-test-inference-contract-v1",
    "prediction_run_id": PREDICTION_RUN_ID,
    "selection_id": SELECTION_ID,
    "selection_seal_sha256": selection_sha256,
    "inference_image_seal": inference_seal_descriptor,
    "inference_image_seal_sha256": inference_seal_sha256,
    "dataset_seal_sha256": dataset_seal_descriptor["sha256"],
    "split_seal_sha256": split_seal_sha256,
    "winner_model_id": selection["winner_model_id"],
    "winner_kind": winner["kind"],
    "checkpoint_bundle_sha256": canonical_sha256(checkpoint_value),
    "routing_sha256": ROUTING_SHA256,
}
INFERENCE_CONTRACT_SHA256 = canonical_sha256(INFERENCE_CONTRACT)
# 같은 v3 test split을 새 dataset version으로 다시 포장해도 두 번째 모델로 평가할 수 없도록
# dataset version이 아니라 upstream split seal SHA를 전역 lease key로 사용합니다.
TEST_USE_LEASE_OBJECT = (
    f"dcc/inference/mission3/test_use_leases/split-{split_seal_sha256}.json"
)
test_use_lease = {
    "status": "TEST_USE_LOCKED_TO_DEPLOYMENT",
    "dataset_version": DATASET_VERSION,
    "split_version": inference_seal["split_version"],
    "split_seal_sha256": split_seal_sha256,
    "prediction_run_id": PREDICTION_RUN_ID,
    "selection_id": SELECTION_ID,
    "selection_seal_sha256": selection_sha256,
    "winner_model_id": selection["winner_model_id"],
    "winner_kind": winner["kind"],
    "checkpoint_bundle_sha256": canonical_sha256(checkpoint_value),
    "routing_sha256": ROUTING_SHA256,
    "inference_contract_sha256": INFERENCE_CONTRACT_SHA256,
}
test_use_lease_descriptor = upload_json_create_only(
    BUCKET,
    test_use_lease,
    TEST_USE_LEASE_OBJECT,
    "m3_test_use_lease",
)
print("winner:", selection["winner_model_id"])
print("dataset version:", DATASET_VERSION)


winner: type-router-convnext-tiny-yolo26s-roi-v1
dataset version: m3ds-v2-pilot


## 2. 익명 shard 검증과 local materialization

tar key에는 `A/B/C/E/F/G/H/S` 첫 문자와 HMAC 24자리 hex만 허용합니다. JSON에는 같은
`sample_id` 하나만 허용하고, 유형은 파일명의 첫 문자에서 파생합니다. 원래 stem 형태나 target
관련 field가 하나라도 발견되면 중단합니다.


In [ ]:
SAFE_ID_PATTERN = __import__("re").compile(r"^[ABCEFGHS][0-9a-f]{24}$")
ALLOWED_META_KEYS = {"sample_id"}
FORBIDDEN_META_KEYS = {
    "stem", "type", "waterlevel_m", "target_m", "clslevel", "archive_key",
    "source_zip_uri", "group_id", "bbox", "obstruction", "annotations",
}

test_shards = []
test_metadata = []
required_bytes = sum(int(item["bytes"]) for item in inference_seal["test_shards"])
if shutil.disk_usage("/content").free < required_bytes + 12 * 1024**3:
    raise RuntimeError("익명 test shard를 받을 local disk가 부족합니다")

for descriptor in inference_seal["test_shards"]:
    if "archive_key" in descriptor:
        raise AssertionError("inference shard descriptor에 archive_key가 있습니다")
    if "/inference/" not in descriptor["uri"] or "/eval_private/" in descriptor["uri"]:
        raise AssertionError("test shard prefix가 안전하지 않습니다")
    _, object_name = parse_gs_uri(descriptor["uri"])
    if not __import__("re").fullmatch(
        rf"dcc/inference/mission3/{__import__('re').escape(DATASET_VERSION)}/"
        r"test_images/test-[0-9a-f]{24}\.tar",
        object_name,
    ):
        raise AssertionError(f"test shard object 이름이 익명 계약을 위반합니다: {object_name}")
    local = LOCAL_SHARDS / f"{descriptor['sha256'][:16]}-{Path(object_name).name}"
    if not local.exists():
        download_object(BUCKET, object_name, local, generation=descriptor["generation"])
    if local.stat().st_size != int(descriptor["bytes"]):
        raise AssertionError("test shard size 불일치")
    if sha256_file(local) != descriptor["sha256"]:
        raise AssertionError("test shard SHA 불일치")
    test_shards.append(str(local))
    with tarfile.open(local, "r:") as handle:
        member_names = {member.name for member in handle.getmembers()}
        keys = sorted({Path(name).stem for name in member_names})
        for key in keys:
            if not SAFE_ID_PATTERN.fullmatch(key):
                raise AssertionError(f"익명 key 형식 위반: {key}")
            if f"{key}.jpg" not in member_names or f"{key}.json" not in member_names:
                raise AssertionError("test tar pair 누락")
            raw_meta = handle.extractfile(f"{key}.json").read()
            metadata = json.loads(raw_meta.decode("utf-8"))
            if set(metadata) != ALLOWED_META_KEYS:
                raise AssertionError(f"test metadata 허용 열 위반: {set(metadata)}")
            if FORBIDDEN_META_KEYS.intersection(metadata):
                raise AssertionError("test metadata에 금지 field가 있습니다")
            if metadata["sample_id"] != key or key[0] not in TYPE_ORDER:
                raise AssertionError("test metadata 값 오류")
            test_metadata.append({"sample_id": key, "type": key[0]})

test_meta_frame = pd.DataFrame(test_metadata).sort_values("sample_id", kind="mergesort")
assert len(test_meta_frame) == EXPECTED_TEST_N
assert test_meta_frame["sample_id"].is_unique
assert test_meta_frame["type"].value_counts().to_dict() == EXPECTED_TEST_TYPES
observed_test_sha = canonical_sha256(sorted(test_meta_frame["sample_id"].tolist()))
assert observed_test_sha == inference_seal["test_sample_id_sha256"]
print("anonymous test contract: PASS", len(test_meta_frame))


anonymous test contract: PASS 30809


In [ ]:
def major_minor(version):
    core = str(version).split("+", 1)[0].split(".")
    return tuple(core[:2])

def verify_training_versions(candidate):
    training_versions = candidate.get("training_package_versions", {})
    for package_name, observed in {
        "torch": torch.__version__, "torchvision": torchvision.__version__
    }.items():
        expected = training_versions.get(package_name)
        if expected and major_minor(expected) != major_minor(observed):
            raise RuntimeError(
                f"{candidate['model_id']} {package_name} 학습/추론 major.minor 불일치: "
                f"train={expected}, infer={observed}"
            )
    if candidate["kind"] != "yolo_roi":
        return
    expected_ultralytics = training_versions.get("ultralytics")
    if expected_ultralytics and str(expected_ultralytics) != str(ultralytics.__version__):
        raise RuntimeError("Ultralytics 학습/추론 버전 불일치")

fullframe_candidate = None
yolo_candidate = None
if winner["kind"] == "full_frame":
    fullframe_candidate = winner
elif winner["kind"] == "yolo_roi":
    yolo_candidate = winner
else:
    fullframe_candidate = winner["components"]["full_frame"]
    yolo_candidate = winner["components"]["yolo_roi"]
    if winner["checkpoint_bundle"]["full_frame"] != fullframe_candidate["checkpoint"]:
        raise AssertionError("router full-frame checkpoint bundle 불일치")
    if winner["checkpoint_bundle"]["yolo_roi"] != yolo_candidate["checkpoint_bundle"]:
        raise AssertionError("router YOLO checkpoint bundle 불일치")
    yolo_fallback = yolo_candidate["checkpoint_bundle"]["fallback"]
    for field in ["uri", "generation", "bytes", "sha256", "artifact_type"]:
        if yolo_fallback[field] != fullframe_candidate["checkpoint"][field]:
            raise AssertionError(f"router fallback checkpoint 불일치: {field}")

for candidate in [fullframe_candidate, yolo_candidate]:
    if candidate is not None:
        verify_training_versions(candidate)

def download_descriptor(
    descriptor, local_name, allowed_artifact_types, expected_prefix
):
    bucket_name, object_name = parse_gs_uri(descriptor["uri"])
    allowed_artifact_types = set(allowed_artifact_types)
    if bucket_name != BUCKET_NAME or not object_name.startswith(expected_prefix):
        raise AssertionError("checkpoint bucket/source run prefix 불일치")
    if descriptor.get("artifact_type") not in allowed_artifact_types:
        raise AssertionError("checkpoint descriptor artifact type 불일치")
    pinned_blob = BUCKET.blob(object_name, generation=int(descriptor["generation"]))
    pinned_blob.reload()
    if (pinned_blob.metadata or {}).get("artifact_type") not in allowed_artifact_types:
        raise AssertionError("checkpoint remote artifact type 불일치")
    local = LOCAL_INPUT / local_name
    download_object(BUCKET, object_name, local, generation=descriptor["generation"])
    if local.stat().st_size != int(descriptor["bytes"]):
        raise AssertionError("checkpoint size 불일치")
    if sha256_file(local) != descriptor["sha256"]:
        raise AssertionError("checkpoint SHA 불일치")
    return local

def safe_meta(value):
    if isinstance(value, (bytes, bytearray)):
        value = json.loads(value.decode("utf-8"))
    if set(value) != ALLOWED_META_KEYS:
        raise AssertionError("inference loader metadata 계약 위반")
    if not SAFE_ID_PATTERN.fullmatch(str(value["sample_id"])):
        raise AssertionError("inference loader sample_id 형식 위반")
    return value

def make_test_loader(transform, batch_size=BATCH_SIZE):
    def mapper(sample):
        image, meta = sample
        meta = safe_meta(meta)
        type_name = str(meta["sample_id"])[0]
        return (
            transform(image),
            torch.tensor(TYPE_TO_INDEX[type_name], dtype=torch.long),
            meta["sample_id"],
            type_name,
        )
    dataset = (
        wds.WebDataset(
            test_shards, shardshuffle=False, empty_check=True,
            handler=wds.handlers.reraise_exception,
        )
        .decode("pil")
        .to_tuple("jpg", "json")
        .map(mapper)
        .batched(batch_size, partial=True)
    )
    return wds.WebLoader(
        dataset, batch_size=None, num_workers=NUM_WORKERS,
        persistent_workers=NUM_WORKERS > 0, pin_memory=True,
    )


## 3A. full-frame 승자 추론

full-frame 승자 또는 type-router의 ConvNeXt component를 전체 익명 test에 실행합니다.
type-router에서는 이 결과가 B/G/S 직접 예측과 YOLO 미검출 fallback에 함께 사용됩니다.


In [ ]:
fullframe_predictions = {}
if fullframe_candidate is not None:
    checkpoint_path = download_descriptor(
        fullframe_candidate["checkpoint"],
        "winner-fullframe.pt",
        {"m3_regressor_checkpoint"},
        expected_prefix=f"dcc/mission3/runs/{fullframe_candidate['source_run_id']}/",
    )
    checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=False)
    if checkpoint.get("schema_version") != "dcc-m3-regressor-checkpoint-v1":
        raise AssertionError("full-frame checkpoint schema 불일치")
    if checkpoint.get("candidate") != fullframe_candidate["backbone"]:
        raise AssertionError("full-frame checkpoint backbone 불일치")
    model = TypeConditionalRegressor(fullframe_candidate["backbone"], pretrained=False)
    model.load_state_dict(checkpoint["model"])
    model.to(DEVICE).eval()
    del checkpoint
    transform = build_transform(int(fullframe_candidate["image_size"]), False, False)
    with torch.inference_mode():
        for images, type_index, sample_ids, types in make_test_loader(transform):
            images = images.to(DEVICE, non_blocking=True)
            type_index = type_index.to(DEVICE, non_blocking=True)
            with torch.cuda.amp.autocast(enabled=True):
                pred = model(images, type_index)
            for sample_id, type_name, value in zip(
                sample_ids, types, pred.float().cpu().numpy()
            ):
                sample_id = str(sample_id)
                if sample_id[0] != str(type_name):
                    raise AssertionError("full-frame type/sample_id 불일치")
                fullframe_predictions[sample_id] = float(value)
    if len(fullframe_predictions) != EXPECTED_TEST_N:
        raise AssertionError("full-frame test prediction 수량 오류")
    del model
    gc.collect()
    torch.cuda.empty_cache()


/tmp/ipykernel_2918/2164929404.py:23: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=True):


## 3B. YOLO26 + ROI 승자 추론

YOLO+ROI 승자 또는 type-router의 YOLO component가 있을 때 실행됩니다. type-router에서는
pilot에서 동결한 A/C/E/F/H 유형만 별도 hard-link 디렉터리에 연결해 detector에 전달합니다.


In [ ]:
def write_bytes_once(path, payload):
    if path.exists():
        if path.stat().st_size != len(payload):
            raise RuntimeError(f"기존 익명 image 충돌: {path}")
        return
    with open(path, "xb") as stream:
        stream.write(payload)

def hardlink_once(source, target):
    if target.exists():
        if not os.path.samefile(source, target):
            raise RuntimeError(f"기존 detector hardlink 충돌: {target}")
        return
    os.link(source, target)

def unpack_anonymous_images():
    metadata_by_id = {}
    for shard_path in test_shards:
        with tarfile.open(shard_path, "r:") as handle:
            members = {member.name: member for member in handle.getmembers()}
            keys = sorted({Path(name).stem for name in members})
            for key in keys:
                if not SAFE_ID_PATTERN.fullmatch(key):
                    raise AssertionError("익명 image key 위반")
                metadata = json.loads(handle.extractfile(members[f"{key}.json"]).read())
                safe_meta(metadata)
                image_bytes = handle.extractfile(members[f"{key}.jpg"]).read()
                write_bytes_once(LOCAL_IMAGES / f"{key}.jpg", image_bytes)
                metadata_by_id[key] = {"sample_id": key, "type": key[0]}
    if len(metadata_by_id) != EXPECTED_TEST_N:
        raise AssertionError("익명 image 추출 수량 오류")
    detector_ids = {
        sample_id for sample_id, metadata in metadata_by_id.items()
        if winner["kind"] != "type_router"
        or route_by_type[metadata["type"]] == "yolo_roi"
    }
    detector_root = (
        LOCAL_ROUTED_DETECTOR_IMAGES
        if winner["kind"] == "type_router" else LOCAL_IMAGES
    )
    if winner["kind"] == "type_router":
        for sample_id in sorted(detector_ids):
            hardlink_once(
                LOCAL_IMAGES / f"{sample_id}.jpg",
                detector_root / f"{sample_id}.jpg",
            )
        if len(detector_ids) != ROUTE_COUNTS["yolo_roi"]:
            raise AssertionError("router YOLO test count 오류")
        observed_detector_ids = {
            path.stem for path in detector_root.glob("*.jpg")
        }
        if observed_detector_ids != detector_ids:
            raise AssertionError("router detector directory ID 집합 오류")
    return metadata_by_id, detector_ids, detector_root

def padded_crop(image, xyxy, padding):
    width, height = image.size
    x1, y1, x2, y2 = map(float, xyxy)
    box_w, box_h = x2 - x1, y2 - y1
    center_x, center_y = (x1 + x2) / 2, (y1 + y2) / 2
    half_w = box_w * (0.5 + padding)
    half_h = box_h * (0.5 + padding)
    px1 = int(max(0, math.floor(center_x - half_w)))
    py1 = int(max(0, math.floor(center_y - half_h)))
    px2 = int(min(width, math.ceil(center_x + half_w)))
    py2 = int(min(height, math.ceil(center_y + half_h)))
    if px2 <= px1 or py2 <= py1:
        raise ValueError("degenerate predicted crop")
    return image.crop((px1, py1, px2, py2))

def weighted_median(values_and_weights):
    ordered = sorted(values_and_weights, key=lambda pair: pair[0])
    total = sum(weight for _, weight in ordered)
    running = 0.0
    for value, weight in ordered:
        running += weight
        if running >= total / 2:
            return float(value)
    return float(ordered[-1][0])

yolo_predictions = {}
yolo_diagnostics = {}
if yolo_candidate is not None:
    metadata_by_id, detector_ids, detector_root = unpack_anonymous_images()
    bundle = yolo_candidate["checkpoint_bundle"]
    yolo_source_prefix = f"dcc/mission3/runs/{yolo_candidate['source_run_id']}/"
    detector_path = download_descriptor(
        bundle["detector"],
        "winner-detector.pt",
        {"m3_yolo26_best_checkpoint", "m3_yolo26_checkpoint"},
        expected_prefix=yolo_source_prefix,
    )
    roi_path = download_descriptor(
        bundle["roi_regressor"],
        "winner-roi.pt",
        {"m3_roi_regressor_checkpoint"},
        expected_prefix=yolo_source_prefix,
    )

    roi_checkpoint = torch.load(roi_path, map_location="cpu", weights_only=False)
    if roi_checkpoint.get("schema_version") != "dcc-m3-roi-checkpoint-v1":
        raise AssertionError("ROI checkpoint schema 불일치")
    if roi_checkpoint.get("roi_config_sha256") != yolo_candidate["roi_config_sha256"]:
        raise AssertionError("ROI checkpoint config 불일치")
    roi_model = TypeConditionalRegressor("convnext_tiny", pretrained=False)
    roi_model.load_state_dict(roi_checkpoint["model"])
    roi_model.to(DEVICE).eval()
    roi_transform = build_transform(
        int(yolo_candidate["roi_config"]["image_size"]), False, False
    )
    del roi_checkpoint

    fallback_model = None
    if winner["kind"] == "type_router":
        fallback_predictions = fullframe_predictions
        if len(fallback_predictions) != EXPECTED_TEST_N:
            raise AssertionError("router full-frame fallback prediction 누락")
    else:
        fallback_path = download_descriptor(
            bundle["fallback"],
            "winner-fallback.pt",
            {"m3_regressor_checkpoint"},
            expected_prefix=f"dcc/mission3/runs/{yolo_candidate['fallback_source_run_id']}/",
        )
        fallback_checkpoint = torch.load(
            fallback_path, map_location="cpu", weights_only=False
        )
        if fallback_checkpoint.get("schema_version") != "dcc-m3-regressor-checkpoint-v1":
            raise AssertionError("fallback checkpoint schema 불일치")
        if fallback_checkpoint.get("candidate") != yolo_candidate["fallback_backbone"]:
            raise AssertionError("fallback checkpoint backbone 불일치")
        fallback_model = TypeConditionalRegressor(
            yolo_candidate["fallback_backbone"], pretrained=False
        )
        fallback_model.load_state_dict(fallback_checkpoint["model"])
        fallback_model.to(DEVICE).eval()
        fallback_transform = build_transform(
            int(yolo_candidate["fallback_image_size"]), False, False
        )
        del fallback_checkpoint

        class AnonymousImageDataset(Dataset):
            def __init__(self, transform):
                self.ids = sorted(metadata_by_id)
                self.transform = transform
            def __len__(self):
                return len(self.ids)
            def __getitem__(self, index):
                sample_id = self.ids[index]
                with Image.open(LOCAL_IMAGES / f"{sample_id}.jpg") as source:
                    image = source.convert("RGB")
                type_name = metadata_by_id[sample_id]["type"]
                return (
                    self.transform(image),
                    torch.tensor(TYPE_TO_INDEX[type_name], dtype=torch.long),
                    sample_id, type_name,
                )

        fallback_predictions = {}
        with torch.inference_mode():
            for images, type_index, sample_ids, _ in DataLoader(
                AnonymousImageDataset(fallback_transform), batch_size=BATCH_SIZE,
                shuffle=False, num_workers=NUM_WORKERS, pin_memory=True,
                persistent_workers=False,
            ):
                images = images.to(DEVICE, non_blocking=True)
                type_index = type_index.to(DEVICE, non_blocking=True)
                with torch.cuda.amp.autocast(enabled=True):
                    pred = fallback_model(images, type_index)
                fallback_predictions.update(
                    zip(map(str, sample_ids), map(float, pred.float().cpu().numpy()))
                )
        if len(fallback_predictions) != EXPECTED_TEST_N:
            raise AssertionError("YOLO fallback prediction 수량 오류")

    detector = YOLO(str(detector_path))
    if isinstance(getattr(detector, "ckpt", None), dict):
        detector.ckpt["optimizer"] = None
        detector.ckpt["scaler"] = None
    crop_predictions = defaultdict(list)
    detection_summary = {}
    crop_tensors, crop_types, crop_ids, crop_weights = [], [], [], []

    def flush_crops():
        if not crop_tensors:
            return
        images = torch.stack(crop_tensors).to(DEVICE)
        type_index = torch.tensor(crop_types, dtype=torch.long, device=DEVICE)
        with torch.inference_mode(), torch.cuda.amp.autocast(enabled=True):
            pred = roi_model(images, type_index).float().cpu().numpy()
        for sample_id, weight, value in zip(crop_ids, crop_weights, pred):
            crop_predictions[sample_id].append((float(value), float(weight)))
        crop_tensors.clear(); crop_types.clear(); crop_ids.clear(); crop_weights.clear()

    detector_stream = detector.predict(
        source=str(detector_root),
        imgsz=int(yolo_candidate["detector_config"]["imgsz"]),
        batch=int(yolo_candidate["detector_config"]["batch"]),
        conf=float(yolo_candidate["inference_rule"]["detector_confidence"]),
        max_det=int(yolo_candidate["inference_rule"]["max_detections"]),
        device=0, stream=True, save=False, verbose=False,
    )
    for index, result in enumerate(detector_stream, start=1):
        sample_id = Path(result.path).stem
        if sample_id not in detector_ids:
            raise AssertionError("detector가 예상하지 않은 파일을 반환했습니다")
        type_name = metadata_by_id[sample_id]["type"]
        boxes = (
            result.boxes.xyxy.detach().cpu().numpy()
            if result.boxes is not None else np.empty((0, 4))
        )
        confidences = (
            result.boxes.conf.detach().cpu().numpy()
            if result.boxes is not None else np.empty(0)
        )
        detection_summary[sample_id] = {
            "detector_conf": float(confidences.max()) if len(confidences) else float("nan"),
            "n_candidates": int(len(boxes)),
            "candidate_boxes_xyxy_json": json.dumps(
                np.asarray(boxes, dtype=float).round(3).tolist(), separators=(",", ":")
            ),
            "candidate_confidences_json": json.dumps(
                np.asarray(confidences, dtype=float).round(6).tolist(), separators=(",", ":")
            ),
        }
        with Image.open(result.path) as source:
            image = source.convert("RGB")
        for box, confidence in zip(boxes, confidences):
            crop = padded_crop(
                image, box, float(yolo_candidate["inference_rule"]["crop_padding"])
            )
            crop_tensors.append(roi_transform(crop))
            crop_types.append(TYPE_TO_INDEX[type_name])
            crop_ids.append(sample_id)
            crop_weights.append(max(float(confidence), 1e-6))
            if len(crop_tensors) >= 64:
                flush_crops()
        if index % 2_000 == 0:
            print("test detector:", index, "/", len(detector_ids))
    flush_crops()
    if set(detection_summary) != detector_ids:
        raise AssertionError("detector test sample 완전성 실패")

    for sample_id in sorted(detector_ids):
        type_name = metadata_by_id[sample_id]["type"]
        crops = crop_predictions.get(sample_id, [])
        fallback_used = not crops
        pred_m = (
            fallback_predictions[sample_id]
            if fallback_used else weighted_median(crops)
        )
        summary = detection_summary[sample_id]
        yolo_predictions[sample_id] = float(pred_m)
        yolo_diagnostics[sample_id] = {
            "detector_conf": summary["detector_conf"],
            "n_candidates": summary["n_candidates"],
            "fallback_used": fallback_used,
            "candidate_boxes_xyxy_json": summary["candidate_boxes_xyxy_json"],
            "candidate_confidences_json": summary["candidate_confidences_json"],
        }
    if set(yolo_predictions) != detector_ids or set(yolo_diagnostics) != detector_ids:
        raise AssertionError("YOLO routed prediction 완전성 실패")
    del detector, detector_stream, roi_model
    if fallback_model is not None:
        del fallback_model
    gc.collect()
    torch.cuda.empty_cache()


/tmp/ipykernel_2918/4234000346.py:189: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.inference_mode(), torch.cuda.amp.autocast(enabled=True):


test detector: 2000 / 23976
test detector: 4000 / 23976
test detector: 6000 / 23976
test detector: 8000 / 23976
test detector: 10000 / 23976
test detector: 12000 / 23976
test detector: 14000 / 23976
test detector: 16000 / 23976
test detector: 18000 / 23976
test detector: 20000 / 23976
test detector: 22000 / 23976


## 4. 예측 hard gate와 create-only seal

예측은 익명 ID 기준 30,809행이어야 하며 모든 값이 finite이고 유형별 물리 범위 안에 있어야
합니다. 이후 private 평가 노트북에서만 원래 stem과 정답을 결합합니다.


In [ ]:
prediction_records = []
diagnostics_records = []
for row in test_meta_frame.itertuples(index=False):
    sample_id = str(row.sample_id)
    type_name = str(row.type)
    if type_name != sample_id[0]:
        raise AssertionError("test type은 sample_id 첫 문자와 같아야 합니다")
    if winner["kind"] == "full_frame":
        route_component = "full_frame"
    elif winner["kind"] == "yolo_roi":
        route_component = "yolo_roi"
    else:
        route_component = route_by_type[type_name]

    if route_component == "full_frame":
        if sample_id not in fullframe_predictions:
            raise AssertionError("full-frame routed prediction 누락")
        pred_m = fullframe_predictions[sample_id]
        diagnostic = {
            "detector_conf": float("nan"),
            "n_candidates": 0,
            "fallback_used": False,
            "candidate_boxes_xyxy_json": "[]",
            "candidate_confidences_json": "[]",
        }
    else:
        if sample_id not in yolo_predictions or sample_id not in yolo_diagnostics:
            raise AssertionError("YOLO routed prediction 누락")
        pred_m = yolo_predictions[sample_id]
        diagnostic = yolo_diagnostics[sample_id]
    prediction_records.append({
        "sample_id": sample_id,
        "type": type_name,
        "pred_m": float(pred_m),
    })
    diagnostics_records.append({
        "sample_id": sample_id,
        "route_component": route_component,
        **diagnostic,
    })

predictions = pd.DataFrame(prediction_records).sort_values(
    "sample_id", kind="mergesort"
).reset_index(drop=True)
diagnostics = pd.DataFrame(diagnostics_records).sort_values(
    "sample_id", kind="mergesort"
).reset_index(drop=True)
assert list(predictions.columns) == ["sample_id", "type", "pred_m"]
assert len(predictions) == len(diagnostics) == EXPECTED_TEST_N
assert predictions["sample_id"].is_unique and diagnostics["sample_id"].is_unique
assert predictions["sample_id"].tolist() == diagnostics["sample_id"].tolist()
assert predictions["sample_id"].tolist() == test_meta_frame["sample_id"].tolist()
assert predictions["type"].value_counts().to_dict() == EXPECTED_TEST_TYPES
assert np.isfinite(predictions["pred_m"].to_numpy(float)).all()
for type_name, (low, high) in TYPE_RANGES.items():
    values = predictions.loc[predictions["type"].eq(type_name), "pred_m"]
    assert values.between(low - 1e-6, high + 1e-6).all()
assert canonical_sha256(predictions["sample_id"].tolist()) == observed_test_sha
if winner["kind"] == "type_router":
    observed_route = predictions["type"].map(route_by_type)
    assert diagnostics["route_component"].tolist() == observed_route.tolist()
    assert diagnostics["route_component"].value_counts().to_dict() == ROUTE_COUNTS
    assert not diagnostics.loc[
        diagnostics["route_component"].eq("full_frame"), "fallback_used"
    ].astype(bool).any()

prediction_path = LOCAL_OUTPUT / "test_predictions.csv"
diagnostics_path = LOCAL_OUTPUT / "test_diagnostics.parquet"
predictions.to_csv(prediction_path, index=False, encoding="utf-8")
diagnostics.to_parquet(diagnostics_path, index=False, compression="zstd")
prediction_descriptor = upload_create_only(
    BUCKET,
    prediction_path,
    f"{ATTEMPT_PREFIX}/test_predictions.csv",
    "m3_anonymous_test_predictions",
)
diagnostics_descriptor = upload_create_only(
    BUCKET,
    diagnostics_path,
    f"{ATTEMPT_PREFIX}/test_diagnostics.parquet",
    "m3_anonymous_test_diagnostics",
)

checkpoint_value = winner.get("checkpoint_bundle", winner.get("checkpoint"))
checkpoint_bundle_sha256 = canonical_sha256(checkpoint_value)
prediction_seal = {
    "status": "TEST_PREDICTIONS_FROZEN",
    "schema_version": "dcc-m3-prediction-seal-v2",
    "prediction_run_id": PREDICTION_RUN_ID,
    "attempt_id": ATTEMPT_ID,
    "attempt_prefix": f"gs://{BUCKET_NAME}/{ATTEMPT_PREFIX}",
    "selection_id": SELECTION_ID,
    "dataset_version": DATASET_VERSION,
    "split_version": inference_seal["split_version"],
    "split_seal_sha256": split_seal_sha256,
    "dataset_seal": dataset_seal_descriptor,
    "winner_model_id": selection["winner_model_id"],
    "winner_kind": winner["kind"],
    "routing_contract": ROUTING_CONTRACT,
    "routing_sha256": ROUTING_SHA256,
    "route_counts": ROUTE_COUNTS,
    "training_scope": selection["training_scope"],
    "row_count": EXPECTED_TEST_N,
    "columns": ["sample_id", "type", "pred_m"],
    "test_sample_id_sha256": observed_test_sha,
    "prediction": prediction_descriptor,
    "diagnostics": diagnostics_descriptor,
    "diagnostics_columns": diagnostics.columns.tolist(),
    "selection_seal_sha256": selection_sha256,
    "inference_image_seal": inference_seal_descriptor,
    "inference_image_seal_sha256": inference_seal_sha256,
    "checkpoint_bundle_sha256": checkpoint_bundle_sha256,
    "inference_contract": INFERENCE_CONTRACT,
    "inference_contract_sha256": INFERENCE_CONTRACT_SHA256,
    "test_use_lease": test_use_lease_descriptor,
    "inference_environment": {
        "python": platform.python_version(),
        "torch": torch.__version__,
        "torchvision": torchvision.__version__,
        "ultralytics": ultralytics.__version__,
        "cuda": torch.version.cuda,
        "gpu": torch.cuda.get_device_name(0),
    },
    "truth_accessed": False,
    "assertions": {
        "row_count_exact": len(predictions) == EXPECTED_TEST_N,
        "sample_ids_unique": bool(predictions["sample_id"].is_unique),
        "sample_id_set_exact": predictions["sample_id"].tolist() == test_meta_frame["sample_id"].tolist(),
        "predictions_finite": bool(np.isfinite(predictions["pred_m"]).all()),
        "type_counts_exact": predictions["type"].value_counts().to_dict() == EXPECTED_TEST_TYPES,
        "route_counts_exact": (
            diagnostics["route_component"].value_counts().to_dict() == ROUTE_COUNTS
            if winner["kind"] == "type_router" else True
        ),
        "route_uses_sample_id_first_character_only": (
            predictions["type"].tolist()
            == predictions["sample_id"].str[0].tolist()
        ),
        "fallback_only_on_yolo_route": bool(
            diagnostics.loc[
                diagnostics["route_component"].eq("full_frame"), "fallback_used"
            ].eq(False).all()
        ),
        "truth_not_accessed": True,
        "private_mapping_not_accessed": True,
    },
}
prediction_seal_descriptor = upload_json_create_only(
    BUCKET,
    prediction_seal,
    f"{OUTPUT_PREFIX}/prediction_seal.json",
    "m3_prediction_seal",
)
print("TEST PREDICTIONS FROZEN")
print("prediction:", prediction_descriptor["uri"])
print("seal:", prediction_seal_descriptor["uri"])
print("truth accessed: False")


TEST PREDICTIONS FROZEN
prediction: gs://YOUR_GCS_BUCKET/dcc/inference/mission3/predictions/m3-test-prediction-router-full-v2/attempts/20260827T231952Z-3ea1cdf1/test_predictions.csv
seal: gs://YOUR_GCS_BUCKET/dcc/inference/mission3/predictions/m3-test-prediction-router-full-v2/prediction_seal.json
truth accessed: False


## 다음 단계

이 runtime을 종료하고 `DCC_Colab_03F_Mission3_Private_RMSE_오류분석.ipynb`을 새 표준
runtime에서 실행합니다. 03F는 prediction SHA-256을 다시 확인한 뒤에만 test truth를 읽습니다.
